### Start by making sure you have your API key setup

In [1]:
import os

if os.getenv("A3_API_KEY", "") == "":
    raise ValueError(
        "A3_API_KEY is not set. Please set it in your environment variables or use `os.environ['A3_API_KEY'] = 'your-8-digit-student-id'`"
    )

### Do imports

In [2]:
import time
import rich

from cs336_scaling.client import (
    get_budget,
    get_experiment,
    list_experiments,
    save_final_submission,
    submit_experiment,
)
from cs336_scaling.training.model.config import BasicTransformerConfig
from cs336_scaling.training.optimizer import AdamWConfig, WarmupCosineDecay
from cs336_scaling.training.training_config import TrainingConfig

/Users/herman/dev/cs336-teach/cs336-monorepo/assignments/scaling/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Start by checking your budget. It should be empty for now

In [3]:
get_budget()

BudgetSummary(used_seconds=0.0, remaining_seconds=43200.0, total_budget_seconds=43200.0)

### 2. Submit an experiment

In [4]:
# define our experiment config
training_config = TrainingConfig(
    architecture_config=BasicTransformerConfig(
        attention_bias=False,
        head_dim=64,
        hidden_size=448,
        intermediate_size=1280,
        num_attention_heads=7,
        num_hidden_layers=9,
        num_key_value_heads=7,
        rms_norm_eps=1e-6,
        rope_theta=1_000_000,
        tie_word_embeddings=False,
        dtype="bfloat16",
        vocab_size=32_000,
    ),
    optimizer_config=AdamWConfig(
        lr_scheduler=WarmupCosineDecay(
            peak_value=3e-4,
            final_lr_frac=0.1,
            warmup_frac=0.05,
            init_value=0.0,
        ),
        weight_decay=1e-2,
        beta1=0.9,
        beta2=0.95,
        eps=1e-8,
        eps_root=1e-8,
        grad_clip_norm=1.0,
    ),
    train_batch_size=128,
    val_batch_size=32,
    n_evals=16,
    total_train_tokens=8_388_608,
    max_runtime_seconds=60.0,
    model_seed=0,
)
rich.print(training_config)

TrainingConfig(
    architecture_config=BasicTransformerConfig(
        attention_bias=False,
        head_dim=64,
        hidden_size=448,
        intermediate_size=1280,
        num_attention_heads=7,
        num_hidden_layers=9,
        num_key_value_heads=7,
        rms_norm_eps=1e-06,
        rope_theta=1000000,
        tie_word_embeddings=False,
        dtype='bfloat16',
        vocab_size=32000
    ),
    optimizer_config=AdamWConfig(
        lr_scheduler=WarmupCosineDecay(peak_value=0.0003, final_lr_frac=0.1, warmup_frac=0.05, init_value=0.0),
        weight_decay=0.01,
        beta1=0.9,
        beta2=0.95,
        eps=1e-08,
        eps_root=1e-08,
        grad_clip_norm=1.0
    ),
    train_batch_size=128,
    val_batch_size=32,
    n_evals=16,
    total_train_tokens=8388608,
    max_runtime_seconds=60.0,
    model_seed=0
)

In [5]:
# submit the experiment
submit_result = submit_experiment(training_config)
submit_result

SubmitResponse(experiment_id=14, budget_summary=BudgetSummary(used_seconds=60.0, remaining_seconds=43140.0, total_budget_seconds=43200.0))

#### You can also submit your config as a JSON-string

The code below will crash, since you've already submitted the same experiment above.

In [6]:
try:
    submit_result_json = submit_experiment(
        {
            "architecture_config": {
                "attention_bias": False,
                "head_dim": 64,
                "hidden_size": 448,
                "intermediate_size": 1280,
                "num_attention_heads": 7,
                "num_hidden_layers": 9,
                "num_key_value_heads": 7,
                "rms_norm_eps": 1e-06,
                "rope_theta": 1000000,
                "tie_word_embeddings": False,
                "dtype": "bfloat16",
                "vocab_size": 32000,
            },
            "optimizer_config": {
                "lr_scheduler": {
                    "peak_value": 0.0003,
                    "final_lr_frac": 0.1,
                    "warmup_frac": 0.05,
                    "init_value": 0.0,
                },
                "weight_decay": 0.01,
                "beta1": 0.9,
                "beta2": 0.95,
                "eps": 1e-08,
                "eps_root": 1e-08,
                "grad_clip_norm": 1.0,
            },
            "train_batch_size": 128,
            "val_batch_size": 32,
            "n_evals": 16,
            "total_train_tokens": 8388608,
            "max_runtime_seconds": 60.0,
            "model_seed": 0,
        }
    )
except RuntimeError as e:
    if "409 Client Error: Conflict" in str(e) and "experiment already exists" in str(e):
        print(
            "Experiment already exists for this training config (expected, continuing)."
        )
        submit_result_json = None
    else:
        raise

Experiment already exists for this training config (expected, continuing).


### Query experiments

In [7]:
# the experiment is in queued state for now
rich.print(list_experiments())

[
    ExperimentResponse(
        experiment_id=14,
        training_config=TrainingConfig(
            architecture_config=BasicTransformerConfig(
                attention_bias=False,
                head_dim=64,
                hidden_size=448,
                intermediate_size=1280,
                num_attention_heads=7,
                num_hidden_layers=9,
                num_key_value_heads=7,
                rms_norm_eps=1e-06,
                rope_theta=1000000,
                tie_word_embeddings=False,
                dtype='bfloat16',
                vocab_size=32000
            ),
            optimizer_config=AdamWConfig(
                lr_scheduler=WarmupCosineDecay(
                    peak_value=0.0003,
                    final_lr_frac=0.1,
                    warmup_frac=0.05,
                    init_value=0.0
                ),
                weight_decay=0.01,
                beta1=0.9,
                beta2=0.95,
                eps=1e-08,
                eps_root=1e-08,
                grad_clip_norm=1.0
            ),
            train_batch_size=128,
            val_batch_size=32,
            n_evals=16,
            total_train_tokens=8388608,
            max_runtime_seconds=60.0,
            model_seed=0
        ),
        status=QueuedExperimentStatus(
            queued_at=datetime.datetime(2026, 4, 29, 23, 46, 36, 972380, tzinfo=TzInfo(0)),
            status_type='queued'
        )
    )
]

In [8]:
# after a few seconds, it will be in running state, as long as the queue is not full
time.sleep(10)
rich.print(list_experiments())

[
    ExperimentResponse(
        experiment_id=14,
        training_config=TrainingConfig(
            architecture_config=BasicTransformerConfig(
                attention_bias=False,
                head_dim=64,
                hidden_size=448,
                intermediate_size=1280,
                num_attention_heads=7,
                num_hidden_layers=9,
                num_key_value_heads=7,
                rms_norm_eps=1e-06,
                rope_theta=1000000,
                tie_word_embeddings=False,
                dtype='bfloat16',
                vocab_size=32000
            ),
            optimizer_config=AdamWConfig(
                lr_scheduler=WarmupCosineDecay(
                    peak_value=0.0003,
                    final_lr_frac=0.1,
                    warmup_frac=0.05,
                    init_value=0.0
                ),
                weight_decay=0.01,
                beta1=0.9,
                beta2=0.95,
                eps=1e-08,
                eps_root=1e-08,
                grad_clip_norm=1.0
            ),
            train_batch_size=128,
            val_batch_size=32,
            n_evals=16,
            total_train_tokens=8388608,
            max_runtime_seconds=60.0,
            model_seed=0
        ),
        status=RunningExperimentStatus(
            queued_at=datetime.datetime(2026, 4, 29, 23, 46, 36, 972380, tzinfo=TzInfo(0)),
            dispatched_at=datetime.datetime(2026, 4, 29, 23, 46, 38, 244101, tzinfo=TzInfo(0)),
            run_id='fc-01KQDT6BXQFMPGVWR3GTS3PVJ3',
            val_losses=[],
            status_type='running'
        )
    )
]

In [9]:
# finally, it will turn into finished state
time.sleep(60)
rich.print(get_experiment(submit_result.experiment_id))

ExperimentResponse(
    experiment_id=14,
    training_config=TrainingConfig(
        architecture_config=BasicTransformerConfig(
            attention_bias=False,
            head_dim=64,
            hidden_size=448,
            intermediate_size=1280,
            num_attention_heads=7,
            num_hidden_layers=9,
            num_key_value_heads=7,
            rms_norm_eps=1e-06,
            rope_theta=1000000,
            tie_word_embeddings=False,
            dtype='bfloat16',
            vocab_size=32000
        ),
        optimizer_config=AdamWConfig(
            lr_scheduler=WarmupCosineDecay(
                peak_value=0.0003,
                final_lr_frac=0.1,
                warmup_frac=0.05,
                init_value=0.0
            ),
            weight_decay=0.01,
            beta1=0.9,
            beta2=0.95,
            eps=1e-08,
            eps_root=1e-08,
            grad_clip_norm=1.0
        ),
        train_batch_size=128,
        val_batch_size=32,
        n_evals=16,
        total_train_tokens=8388608,
        max_runtime_seconds=60.0,
        model_seed=0
    ),
    status=CompletedExperimentStatus(
        queued_at=datetime.datetime(2026, 4, 29, 23, 46, 36, 972380, tzinfo=TzInfo(0)),
        dispatched_at=datetime.datetime(2026, 4, 29, 23, 46, 38, 244101, tzinfo=TzInfo(0)),
        run_id='fc-01KQDT6BXQFMPGVWR3GTS3PVJ3',
        used_runtime_seconds=10.18408550300228,
        val_losses=[
            9.625,
            8.92578125,
            8.37890625,
            7.97265625,
            7.607421875,
            7.4453125,
            7.328125,
            7.291015625,
            7.25,
            7.22265625,
            7.208984375,
            7.19921875,
            7.193359375,
            7.189453125,
            7.189453125,
            7.189453125
        ],
        completed_at=datetime.datetime(2026, 4, 29, 23, 47, 25, 805998, tzinfo=TzInfo(0)),
        status_type='completed'
    )
)

### We are refunded the difference between the budget and the cost of the experiment once it finishes

We reserved 60 seconds for our experiment, but 

In [10]:
get_budget()

BudgetSummary(used_seconds=10.18408550300228, remaining_seconds=43189.815914497, total_budget_seconds=43200.0)

### Submit your final validation. This will run for 48 B200-hours

In [11]:
rich.print(save_final_submission(training_config, 7.2))

FinalSubmissionResponse(
    training_config=TrainingConfig(
        architecture_config=BasicTransformerConfig(
            attention_bias=False,
            head_dim=64,
            hidden_size=448,
            intermediate_size=1280,
            num_attention_heads=7,
            num_hidden_layers=9,
            num_key_value_heads=7,
            rms_norm_eps=1e-06,
            rope_theta=1000000,
            tie_word_embeddings=False,
            dtype='bfloat16',
            vocab_size=32000
        ),
        optimizer_config=AdamWConfig(
            lr_scheduler=WarmupCosineDecay(
                peak_value=0.0003,
                final_lr_frac=0.1,
                warmup_frac=0.05,
                init_value=0.0
            ),
            weight_decay=0.01,
            beta1=0.9,
            beta2=0.95,
            eps=1e-08,
            eps_root=1e-08,
            grad_clip_norm=1.0
        ),
        train_batch_size=128,
        val_batch_size=32,
        n_evals=16,
        total_train_tokens=8388608,
        max_runtime_seconds=60.0,
        model_seed=0
    ),
    predicted_final_loss=7.2,
    submitted_at=datetime.datetime(2026, 4, 29, 23, 47, 47, 218799, tzinfo=TzInfo(0))
)